In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import sys

#from bootcamp_data.config import make_paths

REP_UP = Path("..")
FIGS_DIR = REP_UP / "reports" / 'figures'
#FIGS_DIR = REP_DIR / 


DATA_PARENT = Path("..")
DATA_PROCESSED = DATA_PARENT / 'data' / 'processed'
DATA = DATA_PROCESSED / 'analytics_table.parquet'


FIGS_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fig, path: Path, *, scale: int = 2) -> None:
    """Save a Plotly figure to disk (requires `kaleido`)."""
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.write_image(str(path), scale=scale)

In [2]:
print(DATA)
df = pd.read_parquet(DATA)
df.head(15)

..\data\processed\analytics_table.parquet


,order_id,user_id,amount,quantity,created_at,status,status_clean,quantity__isna,amount__isna,date,year,month,dow,hour,country,signup_date,amount_winsor,amount__is_outlier
0,A0001,0001,12.5,1,2025-12-01 10:05:00+00:00,Paid,paid,False,False,2025-12-01,2025.0,2025-12,Monday,10.0,SA,2025-11-15,12.5,False
1,A0002,0002,8.0,2,2025-12-01 11:10:00+00:00,paid,paid,False,False,2025-12-01,2025.0,2025-12,Monday,11.0,SA,2025-11-20,8.0,False
2,A0003,0003,<NA>,1,2025-12-02 09:00:00+00:00,Refund,refund,False,True,2025-12-02,2025.0,2025-12,Tuesday,9.0,AE,2025-11-22,<NA>,<NA>
3,A0004,0001,25.0,<NA>,2025-12-03 14:30:00+00:00,PAID,paid,True,False,2025-12-03,2025.0,2025-12,Wednesday,14.0,SA,2025-11-15,25.0,False
4,A0005,0004,100.0,1,NaT,paid,paid,False,False,None,NaN,<NA>,None,NaN,SA,2025-11-25,100.0,True
5,A0006,0005,15.75,3,2025-12-04 08:15:00+00:00,Paid,paid,False,False,2025-12-04,2025.0,2025-12,Thursday,8.0,SA,2025-11-26,15.75,False
6,A0007,0002,45.0,1,2025-12-04 12:00:00+00:00,PAID,paid,False,False,2025-12-04,2025.0,2025-12,Thursday,12.0,SA,2025-11-20,45.0,False
7,A0008,0006,<NA>,2,2025-12-05 15:45:00+00:00,Refunded,refunded,False,True,2025-12-05,2025.0,2025-12,Friday,15.0,AE,2025-11-26,<NA>,<NA>
8,A0009,0007,19.99,1,2025-12-06 10:00:00+00:00,paid,paid,False,False,2025-12-06,2025.0,2025-12,Saturday,10.0,SA,2025-11-27,19.99,False
9,A0010,0001,50.0,<NA>,2025-12-07 11:20:00+00:00,paid,paid,True,False,2025-12-07,2025.0,2025-12,Sunday,11.0,SA,2025-11-15,50.0,False


In [3]:
print("raw: ", len(df), "cols: ", len(df.columns))
print(df.dtypes.head(5))

missing = df.isna().sum().sort_values(ascending=False).head(15)

print(missing)

raw:  100 cols:  18
order_id           string[python]
user_id            string[python]
amount                    Float64
quantity                    Int64
created_at    datetime64[ns, UTC]
dtype: object
amount_winsor         12
amount                12
amount__is_outlier    12
quantity               8
hour                   7
dow                    7
month                  7
year                   7
created_at             7
date                   7
user_id                0
order_id               0
status_clean           0
status                 0
amount__isna           0
dtype: int64


## Revenue By Country
As we see, the amount of our SA is more and more amount rather than AE

In [4]:
rev = (
    df.groupby('country', dropna=False)
        .agg(
            n=('order_id', 'size'),
            revenue=('amount', 'sum'),
            aov=('amount', 'mean')
        )
        .reset_index()
        .sort_values('revenue', ascending=False)
)

fig = px.bar(rev, x='country', y='revenue', title="Revenue by country (all data)")
fig.update_layout(title={"x": 0.02})
fig.update_xaxes(title_text="Country")
fig.update_yaxes(title_text="Revenue (sum of amount)")
save_fig(fig, FIGS_DIR / "revenue_by_country.png")
fig


## Revenue Trend By Month
As we see, Revenue go up in the start and end of work days, and drop down in thr middle and the end of weeks

In [13]:
days_order = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]

trend = (
df.groupby("dow", dropna=False)
.agg(n=("order_id","size"), revenue=("amount","sum"))
.reset_index()
#.sort_values("dow")
)

trend["dow"] = pd.Categorical(trend["dow"], categories=days_order, ordered=True)

trend = trend.sort_values("dow")

fig = px.line(trend, x="dow", y="revenue", title="Revenue over time (daily)")
fig.update_layout(title={"x": 0.02})
fig.update_xaxes(title_text="Dow")
fig.update_yaxes(title_text="Revenue")
save_fig(fig, FIGS_DIR / "revenue_trend_daily.png")
fig

## Order Amount distribution
As we see, the majority of orders are for small amounts (around $10-$50), while high-value orders are very rare

In [12]:
fig = px.histogram(df, x="amount_winsor", nbins=30, title="Order amount distribution")
fig.update_layout(title={"x": 0.02})
fig.update_xaxes(title_text="Amount (winsorized)")
fig.update_yaxes(title_text="Number of orders")
save_fig(fig, FIGS_DIR / "amount_hist_winsor.png")
fig

In [14]:
def bootstrap_diff_means(a: pd.Series, b: pd.Series, *, n_boot: int = 2000, seed: int = 0) -> dict:
    rng = np.random.default_rng(seed)
    a = pd.to_numeric(a, errors="coerce").dropna().to_numpy()
    b = pd.to_numeric(b, errors="coerce").dropna().to_numpy()
    
    assert len(a) > 0 and len(b) > 0, "Empty group after cleaning"
    
    diffs = []
    for _ in range(n_boot):
        sa = rng.choice(a, size=len(a), replace=True)
        sb = rng.choice(b, size=len(b), replace=True)
        diffs.append(sa.mean() - sb.mean())
    
    diffs = np.array(diffs)
    
    return {
        "diff_mean": float(a.mean() - b.mean()),
        "ci_low": float(np.quantile(diffs, 0.025)),
        "ci_high": float(np.quantile(diffs, 0.975)),
    }


d = df.assign(is_refund=df["status_clean"].eq("refund").astype(int))

a = d.loc[d["country"].eq("SA"), "is_refund"]
b = d.loc[d["country"].eq("AE"), "is_refund"]

print("n_SA:", len(a), "n_AE:", len(b))

results = bootstrap_diff_means(a, b, n_boot=2000, seed=0)

print(f"Difference (SA - AE): {results['diff_mean']:.4f}")
print(f"95% CI: [{results['ci_low']:.4f}, {results['ci_high']:.4f}]")

n_SA: 76 n_AE: 24
Difference (SA - AE): -0.0482
95% CI: [-0.2281, 0.1053]
